# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the FAIR² dataset package using the `mlcroissant` library. All dataset entities—record sets, fields, and columns—are referenced by their unique `@id` values for clarity and reproducibility.

### Dataset Source
Schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) 

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Loaded: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review the available record sets and their fields. All are referenced by their `@id` for reproducibility.

In [ ]:
# List all record sets (by @id and name) in the dataset
print("Available Record Sets:")
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")

# For demonstration, select the first record set and print its field IDs & names
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    record_set = next(rs for rs in dataset.record_sets if rs['@id'] == selected_record_set_id)
    print(f"\nFields for record set @id {selected_record_set_id}:")
    if 'field' in record_set:
        for fld in record_set['field']:
            print(f"  - @id: {fld['@id']}, name: {fld.get('name', '[no name]')}")

## 3. Data Extraction
Load each record set into a DataFrame for analysis (all by `@id`).

In [ ]:
# Build dataframes for each record set, referencing by @id
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Examine the first dataframe's columns
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Explore, process, and group the data by applying filters on numeric columns, normalization, categorization, and groupby aggregation. All fields and columns are referenced via their `@id`.

In [ ]:
# Choose a numeric field for filtering and normalization, referencing by @id
# We'll auto-detect one numeric field for demonstration. Adjust as needed per dataset docs.
import numpy as np

df = dataframes.get(first_rs_id)
numeric_field_id = None
for col in df.columns:
    if np.issubdtype(df[col].dtype, np.number):
        numeric_field_id = col
        break

if numeric_field_id:
    print(f"Using numeric field '@id': {numeric_field_id}\n")
    threshold = df[numeric_field_id].mean()  # Filter by mean for demo
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records")
    
    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    display(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Try grouping by a non-numeric field (auto-select first object/string field)
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and (df[col].dtype == object or str(df[col].dtype).startswith('string')):
            group_field_id = col
            break
    if group_field_id:
        print(f"\nGrouping filtered data by field '@id': {group_field_id} and computing mean of numeric fields.")
        grouped = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().to_frame()
        grouped = grouped.rename(columns={numeric_field_id: f"{numeric_field_id}_mean"})
        display(grouped.head())
else:
    print("No numeric field auto-detected in record set. Please adjust the code to specify a numeric column '@id'.")

## 5. Visualization
Visualize the distributions and relationships. You can interact with any numeric and groupable field using their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have successfully loaded and explored the FAIR² dataset using `mlcroissant`, referenced all data structures by their unique `@id`, and performed initial data wrangling and visualization. This approach ensures robustness and reproducibility for future dataset versions and analyses. For further insight, consult the [Croissant schema specification](https://mlcommons.org/croissant/) and the dataset documentation.